In [3]:
import gzip
import csv
import pandas as pd

In [4]:
matrix_path = "../data/raw/GSE103001_series_matrix.txt.gz"

In [6]:
metadata = {}

with gzip.open(matrix_path, "rt") as f:
    reader = csv.reader(f, delimiter="\t")

    for row in reader:
        if row and row[0].startswith("!Sample_"):
            metadata.setdefault(row[0], []).append(row[1:])

print("Metadata fields: ", len(metadata))
print("Samples: ", len(metadata["!Sample_geo_accession"][0]))

Metadata fields:  31
Samples:  44


In [7]:
for characteristic in metadata["!Sample_characteristics_ch1"]:
    print(
        characteristic[0],
        "|",
        characteristic[22]
    )

tissue nature: normal | tissue nature: primary invasive breast carcinoma
tumor receptors: / | tumor receptors: ER+/HER2-
bloom: / | bloom: 2
er subtype: / | er subtype: A
tnm: / | tnm: T2N1aM0


In [14]:
def classify_condition(tissue):
    tissue = tissue.lower()

    if tissue == "normal":
        return "normal"
    elif tissue == "primary invasive breast carcinoma":
        return "tumor"
    else:
        raise ValueError(f"Unexpected tissue type: {tissue}")

In [19]:
titles = metadata["!Sample_title"][0]
accessions = metadata["!Sample_geo_accession"][0]

# Create a DataFrame with accession and title columns
sample_df = pd.DataFrame({
    "accession": accessions,
    "title": titles
})

# Add sample characteristics to the DataFrame
for characteristic in metadata["!Sample_characteristics_ch1"]:
    key = characteristic[0].split(":")[0].strip()

    # Extract the value after the colon and strip whitespace
    values = [
        value.split(":", 1)[1].strip()
        for value in characteristic
    ]

    sample_df[key] = values

# Add a new column "condition" based on the "tissue nature" column
print(sample_df["tissue nature"].unique())
sample_df["condition"] = (
    sample_df["tissue nature"].apply(classify_condition)
)

# Extract patient ID from title, e.g. "Pat_12-02_tumor" -> "12-02"
sample_df["patient"] = (
    sample_df["title"]
    .str.replace(r"^Pat_", "", regex=True)
    .str.replace(r"_(normal|tumor)$", "", regex=True)
)

sample_df.head()

<StringArray>
['normal', 'primary invasive breast carcinoma']
Length: 2, dtype: str


,accession,title,tissue nature,tumor receptors,bloom,er subtype,tnm,condition,patient
0,GSM2752350,Pat_12-02_normal,normal,/,/,/,/,normal,12-02
1,GSM2752351,Pat_12-03_normal,normal,/,/,/,/,normal,12-03
2,GSM2752352,Pat_13-02_normal,normal,/,/,/,/,normal,13-02
3,GSM2752353,Pat_13-03_normal,normal,/,/,/,/,normal,13-03
4,GSM2752354,Pat_13-05_normal,normal,/,/,/,/,normal,13-05


In [21]:
tumors = sample_df[sample_df["condition"] == "tumor"]

print("Tumor samples: ", len(tumors))
print("\nReceptor status: ")
print(tumors["tumor receptors"].value_counts(dropna=False))

print("\nER subtype: ")
print(tumors["er subtype"].value_counts(dropna=False))

print("\nPatients with both conditions: ")
print(
    sample_df.groupby("patient")["condition"].nunique().value_counts()
)

Tumor samples:  22

Receptor status: 
tumor receptors
ER+/HER2-    22
Name: count, dtype: int64

ER subtype: 
er subtype
A    12
B    10
Name: count, dtype: int64

Patients with both conditions: 
condition
2    22
Name: count, dtype: int64
